In [ ]:
import zipfile
import os
# 론잡도 데이터 압축 해제
with zipfile.ZipFile('/content/2024_역별_승하차_혼잡도.zip', 'r') as zipf:   # 2024 데이터의 경우
    zipf.extractall('/content/good')
    
import pandas as pd

def process_single_file(in_path, out_path):
    # 파일 로드
    df_single = pd.read_csv(in_path)
    df_base = pd.read_csv("/content/병합용_베이스_데이터.csv")

    # [1] 단일 파일 전처리
    df_single_trimmed = df_single[['날짜', 'hour', '호선', 'inner_congestion_pct', 'outer_congestion_pct']].copy()
    df_single_trimmed['Datetime'] = pd.to_datetime(df_single_trimmed['날짜'] + ' ' + df_single_trimmed['hour'].astype(str) + ':00')
    df_single_trimmed = df_single_trimmed.rename(columns={'호선': 'Line'})

    # 기준 파일 시간 정리
    df_base['Datetime'] = pd.to_datetime(df_base['Datetime'])

    # 병합
    df_merged = pd.merge(
        df_base,
        df_single_trimmed[['Datetime', 'Line', 'inner_congestion_pct', 'outer_congestion_pct']],
        on='Datetime',
        how='left',
        suffixes=('', '_filled')
    )

    # NaN만 채우기
    for col in ['Line', 'inner_congestion_pct', 'outer_congestion_pct']:
        fill_col = f"{col}_filled"
        df_merged[col] = df_merged[col].where(~df_merged[col].isna(), df_merged[fill_col])
        df_merged.drop(columns=[fill_col], inplace=True)

    # 정렬 및 시간 시프트 변수 생성
 
    df_merged = df_merged.sort_values(by='Datetime')
    df_merged['prev_1h_inner_congestion_pct'] = df_merged['inner_congestion_pct'].shift(2)
    df_merged['prev_1h_outer_congestion_pct'] = df_merged['outer_congestion_pct'].shift(2)
    df_merged['prev_24h_inner_congestion_pct'] = df_merged['inner_congestion_pct'].shift(48)
    df_merged['prev_24h_outer_congestion_pct'] = df_merged['outer_congestion_pct'].shift(48)

    # 시계열 시작부 결측 보정 (순환 구조 기반)
    df_merged.loc[df_merged['prev_1h_inner_congestion_pct'].isna(), 'prev_1h_inner_congestion_pct'] = df_merged['inner_congestion_pct'].iloc[-2]
    df_merged.loc[df_merged['prev_1h_outer_congestion_pct'].isna(), 'prev_1h_outer_congestion_pct'] = df_merged['outer_congestion_pct'].iloc[-2]

    # 정확히 앞 24행을 뒤 24시간의 값으로 채우기
    # 단일, 다중에 따라 24 or 48로 바뀜
    total_len = len(df_merged)
    inner_idx = df_merged.columns.get_loc('inner_congestion_pct')
    outer_idx = df_merged.columns.get_loc('outer_congestion_pct')
    prev_inner_idx = df_merged.columns.get_loc('prev_24h_inner_congestion_pct')
    prev_outer_idx = df_merged.columns.get_loc('prev_24h_outer_congestion_pct')

    df_merged.iloc[0:48, prev_inner_idx] = df_merged.iloc[total_len-48:total_len, inner_idx].values
    df_merged.iloc[0:48, prev_outer_idx] = df_merged.iloc[total_len-48:total_len, outer_idx].values

    # 저장
    df_merged.to_csv(out_path, index=False)
    
import os
for name in station2:
    in_path = f"/content/good/{name}_내외선_승하차_혼잡도.csv"
    out_path = f"/content/output_2/processed_{name}.csv"  # 출력 파일명

    try:
        process_single_file(in_path, out_path)
        print(f"처리 완료: {name}")
    except FileNotFoundError:
        print(f"파일 없음: {name}")